# EMO addressable population — segmented EDA

**Audience:** Growth / marketing outreach (EMO second-opinion targeting)

## Questions this notebook answers

1. **Segmented EDA:** Clients **with claims** vs **without claims** (CY2025 parent EMO accounts).
2. **Outreach member list (Allison Kim):** Members with any of `COMPLEX_PATIENTS`, `MSK_NEURO`, `HIGH_COST` while eligible in the **same calendar month** (`member_risk_assessments.history:v1`).
3. **With-claims strategy:** Among claims clients, how many **outreach-list** members also have **paid claims history** (claims are *not* in Allison’s selection logic; we test adding that filter).
4. **Without-claims strategy:** Benchmark addressable % from **with-claims** clients; extrapolate to no-claims clients (directional).
5. **Logic verification:** Audit whether claims are used in outreach identification (Allison list + *EMO Segments for outreach* doc); see Step 8 and `docs/emo-outreach-logic-verification.md`.

**Prerequisite:** In a terminal:

```bash
aws-environment production enduser_yellow_production
```

Run with cwd = `modules/standard_reports/project_scrappy/`.

## Step 1: Connect to Lasik

In [21]:
import os
from datetime import date
from pathlib import Path

import laaso
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.environ.setdefault('AWS_ENVIRONMENT', 'production')

client = laaso.Client(catalog_type=laaso.CatalogType.YELLOW, env='production')
dialect = laaso.SqlDialect.PRESTO

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'queries' / 'emo_nav_br' / 'emo_addressable_eda.sql').exists():
    NOTEBOOK_DIR = Path.cwd() / 'modules' / 'standard_reports' / 'project_scrappy'

QUERY_PATH = NOTEBOOK_DIR / 'queries' / 'emo_nav_br' / 'emo_addressable_eda.sql'
EXPORT_DIR = NOTEBOOK_DIR / 'query_data' / 'emo_addressable'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Notebook dir:', NOTEBOOK_DIR.resolve())

Notebook dir: /Users/john.kimutai/customerinsights-analytics/modules/standard_reports/project_scrappy


## Step 2: Parameters

In [22]:
REPORT_START = '2025-01-01'
REPORT_END = '2026-01-01'  # end-exclusive (CY2025)
CLAIMS_TRUST_LOOKBACK = '2024-01-01'
RUN_DATE = date.today().strftime('%Y%m%d')

# Allison Kim — EMO marketing outreach selection logic (members matching ANY code)
OUTREACH_RISK_ASSESSMENTS = [
    'COMPLEX_PATIENTS',
    'MSK_NEURO',
    'HIGH_COST',
]

PARAMS = {
    'report_start_date': REPORT_START,
    'report_end_date': REPORT_END,
    'claims_trust_lookback_date': CLAIMS_TRUST_LOOKBACK,
}

## Step 3: Load warehouse data

In [ ]:
def load_sql(final_cte: str) -> str:
    sql = QUERY_PATH.read_text()
    for key, val in PARAMS.items():
        sql = sql.replace('{' + key + '}', val)
    return sql.format(final_cte=final_cte)


def run_query(final_cte: str) -> pd.DataFrame:
    q = load_sql(final_cte)
    print(f'Running final_cte={final_cte}...')
    return client.pandas.query(q, dialect)


df_clients = run_query('client_claims_profile')
df_client_eda = run_query('client_eda')
df_outreach_list = run_query('outreach_member_list')
assert set(df_outreach_list['outreach_riskassessment'].unique()).issubset(
    set(OUTREACH_RISK_ASSESSMENTS)
)
print('Clients:', len(df_clients))
print('Outreach member-rows:', len(df_outreach_list))
print('Distinct outreach members:', df_outreach_list['entity_id'].nunique())

Running final_cte=client_claims_profile...


Running final_cte=client_eda...
Running final_cte=outreach_member_list...


## Step 4: Segmented EDA — clients with vs without claims

In [ ]:
workstream_summary = (
    df_client_eda.groupby('client_workstream', dropna=False)
    .agg(
        n_clients=('client_account_id', 'count'),
        total_outreach_members=('n_outreach_members', 'sum'),
        total_addressable_risk_claims=('n_addressable_risk_and_claims_history', 'sum'),
        median_pct_on_outreach_list=('pct_eligible_on_outreach_list', 'median'),
        median_pct_addressable=('pct_addressable_risk_and_claims', 'median'),
    )
    .reset_index()
    .sort_values('n_clients', ascending=False)
)
display(workstream_summary)

claims_present = df_client_eda['is_claims_present'].value_counts().rename_axis('is_claims_present').reset_index(name='n_clients')
display(claims_present)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ws = workstream_summary.sort_values('n_clients')
ax.barh(ws['client_workstream'], ws['n_clients'], color='#0012E7')
ax.set_xlabel('Number of EMO parent accounts')
ax.set_title('EMO parent accounts: with claims vs without claims (CY2025)')
plt.tight_layout()
plt.show()

## Step 5: With-claims cohort — risk vs risk + paid claims history

Allison’s outreach list = **risk codes only** (above). **Addressable (strict)** = outreach-list members with **paid claims** in CY2025 — the proposed add-on for with-claims clients.

In [ ]:
with_claims = df_client_eda[df_client_eda['is_claims_present'] == True].copy()  # noqa: E712

with_claims['pct_uplift_from_claims_filter'] = (
    with_claims['pct_addressable_risk_and_claims']
    / with_claims['pct_eligible_on_outreach_list'].replace(0, np.nan)
)

with_claims_summary = pd.DataFrame({
    'metric': [
        'clients',
        'outreach_list_members',
        'addressable_risk_and_claims',
        'median_pct_eligible_on_outreach_list',
        'median_pct_addressable_risk_and_claims',
    ],
    'value': [
        len(with_claims),
        with_claims['n_outreach_members'].sum(),
        with_claims['n_addressable_risk_and_claims_history'].sum(),
        with_claims['pct_eligible_on_outreach_list'].median(),
        with_claims['pct_addressable_risk_and_claims'].median(),
    ],
})
display(with_claims_summary)

top_delta = with_claims.assign(
    member_delta=with_claims['n_outreach_members']
    - with_claims['n_addressable_risk_and_claims_history']
).sort_values('member_delta', ascending=False).head(15)[
    [
        'client_account_name',
        'client_workstream',
        'n_outreach_members',
        'n_addressable_risk_and_claims_history',
        'member_delta',
        'pct_eligible_on_outreach_list',
        'pct_addressable_risk_and_claims',
    ]
]
display(top_delta)

## Step 6: Benchmark & extrapolation (without-claims clients)

Benchmark = distribution of `pct_addressable_risk_and_claims` among **WITH_CLAIMS** clients. Apply median (and p25–p75 band) to **WITHOUT_CLAIMS** clients.

In [ ]:
benchmark_clients = df_client_eda[
    df_client_eda['client_workstream'] == 'WITH_CLAIMS'
].copy()

if benchmark_clients.empty:
    raise ValueError('No WITH_CLAIMS clients — check claims presence logic.')

bench_median = benchmark_clients['pct_addressable_risk_and_claims'].median()
bench_p25 = benchmark_clients['pct_addressable_risk_and_claims'].quantile(0.25)
bench_p75 = benchmark_clients['pct_addressable_risk_and_claims'].quantile(0.75)

print(f'Benchmark clients: {len(benchmark_clients)}')
print(f'Median addressable %: {bench_median:.2%}  (p25={bench_p25:.2%}, p75={bench_p75:.2%})')

no_claims = df_client_eda[df_client_eda['client_workstream'] == 'WITHOUT_CLAIMS'].copy()

no_claims['est_addressable_median'] = (
    no_claims['n_eligible_distinct_members'] * bench_median
).round(0)
no_claims['est_addressable_p25'] = (no_claims['n_eligible_distinct_members'] * bench_p25).round(0)
no_claims['est_addressable_p75'] = (no_claims['n_eligible_distinct_members'] * bench_p75).round(0)

extrapolation_summary = pd.DataFrame({
    'metric': ['no_claims_clients', 'eligible_members_no_claims', 'est_addressable_median_total'],
    'value': [
        len(no_claims),
        no_claims['n_eligible_distinct_members'].sum(),
        no_claims['est_addressable_median'].sum(),
    ],
})
display(extrapolation_summary)
display(
    no_claims.sort_values('n_eligible_distinct_members', ascending=False)[
        [
            'client_account_name',
            'n_eligible_distinct_members',
            'n_outreach_members',
            'est_addressable_median',
            'est_addressable_p25',
            'est_addressable_p75',
        ]
    ].head(20)
)

## Step 7: Recommended client-level export

In [ ]:
df_export = df_client_eda.copy()
df_export['benchmark_median_pct'] = bench_median
df_export['benchmark_p25_pct'] = bench_p25
df_export['benchmark_p75_pct'] = bench_p75

df_export['n_addressable_recommended'] = np.where(
    df_export['is_claims_present'],
    df_export['n_addressable_risk_and_claims_history'],
    (df_export['n_eligible_distinct_members'] * bench_median).round(0),
)
df_export['pct_addressable_recommended'] = np.where(
    df_export['is_claims_present'],
    df_export['pct_addressable_risk_and_claims'],
    bench_median,
)

out_path = EXPORT_DIR / f'emo_addressable_client_eda_{REPORT_START}_{REPORT_END}_{RUN_DATE}.csv'
df_export.to_csv(out_path, index=False)
print('Wrote', out_path)

outreach_path = EXPORT_DIR / f'emo_outreach_member_list_{REPORT_START}_{REPORT_END}_{RUN_DATE}.csv'
df_outreach_list.to_csv(outreach_path, index=False)
print('Wrote', outreach_path)

df_export.sort_values('n_addressable_recommended', ascending=False).head(10)

## Step 8: Logic verification — are claims used in outreach identification?

Audits Allison’s list against *EMO Segments for outreach* (see `docs/emo-outreach-logic-verification.md`).

**Layer A — Outreach rule:** OR of three `riskassessment` codes + eligibility. No separate claims join.

**Layer B — Flag definitions:** `HIGH_COST` and `MSK_NEURO` require claims; `COMPLEX_PATIENTS` is mostly claims-driven (≥3 of 16 factors) with some auth paths.

This step **documents** Layer A/B and **validates** Layer B in warehouse data (flag mix on with- vs without-claims clients).

In [ ]:
# --- Layer A: outreach selection rule (Allison) ---
SELECTION_AUDIT = pd.DataFrame([
    {
        'layer': 'A_outreach_selection',
        'question': 'Separate claims join in outreach rule?',
        'answer': 'No',
        'detail': 'Outreach = riskassessment IN (COMPLEX_PATIENTS, MSK_NEURO, HIGH_COST) OR + eligible',
    },
    {
        'layer': 'B_HIGH_COST',
        'question': 'Claims required to assign flag?',
        'answer': 'Yes',
        'detail': 'Med+Rx spend thresholds only (EMO Segments doc)',
    },
    {
        'layer': 'B_MSK_NEURO',
        'question': 'Claims required to assign flag?',
        'answer': 'Yes',
        'detail': 'Qualifying medical claim in last 60 days (dx/procedure pathways)',
    },
    {
        'layer': 'B_COMPLEX_PATIENTS',
        'question': 'Claims required to assign flag?',
        'answer': 'Mostly',
        'detail': '>=3 of 16 contributors; mostly claims/dx/Rx/IP/ER; some auth/ADT',
    },
])
display(SELECTION_AUDIT)

flag_breakdown = (
    df_outreach_list.groupby('outreach_riskassessment')['entity_id']
    .nunique()
    .reset_index(name='n_members_portfolio')
    .sort_values('n_members_portfolio', ascending=False)
)
flag_breakdown['claims_required_per_segment_doc'] = flag_breakdown['outreach_riskassessment'].map({
    'HIGH_COST': 'Yes — claims/spend only',
    'MSK_NEURO': 'Yes — medical claims (60d)',
    'COMPLEX_PATIENTS': 'Mostly — >=3 of 16; some auth',
})
display(flag_breakdown)

# --- Layer B: empirical — outreach on clients without claims feed ---
outreach_with_flags = df_outreach_list.merge(
    df_client_eda[['client_account_id', 'client_workstream', 'is_claims_present']],
    on='client_account_id',
    how='left',
)

empirical_by_workstream = (
    outreach_with_flags.groupby('client_workstream')
    .agg(
        n_clients=('client_account_id', 'nunique'),
        n_outreach_member_rows=('entity_id', 'count'),
        n_distinct_members=('entity_id', 'nunique'),
    )
    .reset_index()
)
display(empirical_by_workstream)

no_claims_flags = (
    outreach_with_flags.loc[
        outreach_with_flags['client_workstream'] == 'WITHOUT_CLAIMS'
    ]
    .groupby('outreach_riskassessment')['entity_id']
    .nunique()
    .reset_index(name='n_members_no_claims_clients')
)
display(no_claims_flags)

with_claims_only = df_client_eda[df_client_eda['is_claims_present'] == True].copy()  # noqa: E712
with_claims_only['pct_outreach_with_paid_claims_cy2025'] = (
    with_claims_only['n_addressable_risk_and_claims_history']
    / with_claims_only['n_outreach_members'].replace(0, np.nan)
)

portfolio_pct_on_paid_claims = (
    with_claims_only['n_addressable_risk_and_claims_history'].sum()
    / with_claims_only['n_outreach_members'].sum()
)
print(
    f"Portfolio (with-claims clients): {portfolio_pct_on_paid_claims:.1%} of outreach-list "
    'members also have paid claims in CY2025 (overlap check; flags already claims-derived).'
)

LOGIC_VERIFICATION_SUMMARY = pd.DataFrame([
    {
        'finding': 'explicit_claims_in_outreach_rule',
        'value': 'No',
        'evidence': 'Allison list = three riskassessment codes only',
    },
    {
        'finding': 'claims_used_in_flag_assignment',
        'value': 'Yes (HIGH_COST, MSK_NEURO); Mostly (COMPLEX_PATIENTS)',
        'evidence': 'EMO Segments for outreach doc',
    },
    {
        'finding': 'n_outreach_members_portfolio',
        'value': df_outreach_list['entity_id'].nunique(),
        'evidence': 'emo_outreach_member_list',
    },
    {
        'finding': 'n_outreach_on_without_claims_clients',
        'value': outreach_with_flags.loc[
            outreach_with_flags['client_workstream'] == 'WITHOUT_CLAIMS', 'entity_id'
        ].nunique(),
        'evidence': 'Should be low if HIGH_COST/MSK_NEURO require claims feed',
    },
    {
        'finding': 'pct_outreach_with_paid_claims_cy2025_with_claims_clients',
        'value': round(portfolio_pct_on_paid_claims, 4),
        'evidence': 'Empirical overlap; not a definition of outreach',
    },
])
display(LOGIC_VERIFICATION_SUMMARY)

verify_path = EXPORT_DIR / (
    f'emo_outreach_logic_verification_{REPORT_START}_{REPORT_END}_{RUN_DATE}.csv'
)
LOGIC_VERIFICATION_SUMMARY.to_csv(verify_path, index=False)
print('Wrote', verify_path)
print(
    '\nAudit conclusion: Claims are NOT a separate outreach filter (Layer A), but ARE '
    'used to identify most outreach members via risk-assessment definitions (Layer B). '
    'See docs/emo-outreach-logic-verification.md for full write-up.'
)

## Step 9: QA checks

In [ ]:
assert df_client_eda['client_account_id'].is_unique, 'Duplicate client_account_id rows'
assert df_client_eda['n_outreach_members'].ge(0).all()
assert (
    df_client_eda['n_addressable_risk_and_claims_history']
    <= df_client_eda['n_outreach_members']
).all(), 'Addressable should not exceed outreach list count'

# Per-client outreach counts should match member list at (client, entity) grain.
# Summing across clients can exceed portfolio-unique members when one entity_id
# appears under more than one parent account in the EMO book.
outreach_by_client = (
    df_outreach_list.groupby('client_account_id')['entity_id']
    .nunique()
    .rename('n_outreach_from_list')
)
outreach_qa = df_client_eda.merge(
    outreach_by_client,
    on='client_account_id',
    how='left',
    validate='one_to_one',
)
assert outreach_qa['n_outreach_from_list'].fillna(0).astype(int).equals(
    outreach_qa['n_outreach_members'].astype(int)
), 'client_eda outreach counts do not match outreach_member_list'

client_entity_slots = outreach_qa['n_outreach_members'].sum()
portfolio_unique_members = df_outreach_list['entity_id'].nunique()
cross_client_duplicates = client_entity_slots - portfolio_unique_members

print('QA passed.')
print('Period:', REPORT_START, 'to', REPORT_END)
print('Parent EMO clients:', len(df_client_eda))
print('Outreach member slots (sum per client):', f'{client_entity_slots:,}')
print('Portfolio-unique outreach members:', f'{portfolio_unique_members:,}')
if cross_client_duplicates:
    print(
        'Note:',
        f'{cross_client_duplicates:,}',
        'member slot(s) are entity_ids counted under more than one client.',
    )